[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/27.%20The%20Integrated%20Berth%20%26%20Crane%20Allocation%20Problem%20%28BAP-QCAP%29/P27-Tier-7_executed.ipynb)

---


# 27. The Integrated Berth and Crane Allocation Problem (BAP-QCAP)
## Tier 7: Human–AI Collaboration

### Purpose and scope

This notebook demonstrates a **human–AI collaboration workflow** for berth and quay-crane recommendations. The AI component uses a transparent, rule-based greedy procedure: it orders vessels by priority and arrival time, assigns a berth position using a simple location preference, chooses a crane count by evaluating effective productivity, and schedules service sequentially. A simulated human expert can then suggest contextual modifications. The notebook reports illustrative performance indicators and records outcomes across repeated sessions.

This is an educational prototype, not a validated port-planning system and not an exact solution to the full BAP-QCAP. In particular, the code does not model simultaneous vessel service, time-dependent berth occupancy, crane travel or interference between vessels, tides, safety clearances, labor rosters, or uncertainty in processing times. Human modifications are simulated rather than supplied by an operator, and the performance measures are proxies rather than measured operational outcomes. Recommendations therefore require operational review before use.

### Inputs and outputs

The inputs are vessel length, workload, arrival time, draft, priority, and a customer-value score, together with quay length, crane availability, crane productivity, and interference assumptions. The main output is a sequence of AI recommendations, optionally modified by the simulated human expert. Each recommendation includes a berth position, service start time, crane count, a heuristic confidence score, an explanation, alternatives, and estimated performance measures.

### Concepts used in the procedure

- **Greedy procedure:** a sequential rule that makes a choice for the current vessel using the rules defined in the code. It does not enumerate all possible schedules or guarantee a globally optimal plan.
- **Effective productivity:** the estimated handling rate after accounting for crane interference and an additional penalty when more than three cranes are assigned.
- **Sequential schedule:** each vessel starts no earlier than its arrival and the completion of the previous vessel. This assumption allows a berth position to be reused because vessels do not overlap in time in this prototype.
- **Human modification:** a simulated contextual suggestion. The collaborative schedule reapplies the suggestions in sequence and respects vessel arrival times and the preceding service completion.
- **Confidence and performance scores:** heuristic indicators defined by this notebook. They are not statistically calibrated confidence probabilities or empirical comparisons of human and AI performance.

### Mathematical representation

Let \(V\) be the set of vessels. The procedure orders vessels using priority and arrival time, then processes them one at a time. For vessel \(i\), let \(a_i\) be its arrival time, \(w_i\) its workload, \(k_i\) its assigned crane count, and \(p_i\) its berth position. The service duration used by the schedule is

$$
t_i = \frac{w_i}{q k_i\left(1-\alpha(k_i-1)\right)\delta(k_i)},
$$

where \(q\) is productivity per crane, \(\alpha\) is the interference factor, and

$$
\delta(k_i) =
\begin{cases}
1, & k_i \leq 3,\\
0.9, & k_i > 3.
\end{cases}
$$

If vessels are processed in the order \(i_1, i_2, \ldots, i_{|V|}\), the sequential schedule is

$$
s_{i_1}=a_{i_1}, \qquad
s_{i_j}=\max\left(a_{i_j},\ s_{i_{j-1}}+t_{i_{j-1}}\right) \quad (j>1).
$$

Here \(s_i\) is the service start time. In the code, these quantities correspond to `service_time`, `start_time`, `current_time`, and `crane_count`. Crane allocation evaluates each allowed integer crane count and selects the one with the highest effective productivity; it does not optimize a complete vessel schedule. Berth position selection evaluates 50-meter candidate positions and uses a draft-related preference. Because service is sequential, previous vessels do not occupy the berth when the next vessel starts.

For reporting, the notebook maps a total-time proxy \(T\) to a score

$$
P = \frac{100}{1+T}.
$$

This score is a convenient monotone transformation for the demonstration, not a standard BAP-QCAP objective. The model does not claim that maximizing this score yields an operationally optimal plan.

The sequential pass requires work proportional to the number of vessels times the number of candidate berth positions and crane counts. It does not search the combinatorial space of all possible joint berth, crane, and timing plans.

### Workflow

1. Define the vessel and recommendation data structures.
2. Generate explainable AI recommendations using the greedy rules.
3. Simulate a human review and record any contextual modifications.
4. Rebuild the sequential schedule with the accepted modifications.
5. Report proxy metrics, inspect recommendations, and visualize results.
6. Repeat sessions to record observed metric changes. Recording a history is not, by itself, a learning algorithm; this prototype does not update its recommendation rules from feedback.


In [ ]:
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fixed seeds make the simulated human decisions and arrival-time variations reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for the human–AI collaboration demonstration.")

In [ ]:
@dataclass
class Vessel:
    """Input data for one vessel in the BAP-QCAP demonstration."""

    id: int
    name: str
    length: float  # meters
    workload: float  # TEU
    arrival_time: float  # hours from the planning horizon start
    draft: float  # meters
    priority: int  # 1 = high, 2 = medium, 3 = low
    customer_value: float  # illustrative score from 0 to 1


@dataclass
class AIRecommendation:
    """AI recommendation with a heuristic confidence score and explanation."""

    vessel_id: int
    berth_position: float
    start_time: float
    crane_count: int
    confidence: float
    reasoning: List[str]
    alternatives: List[Dict]
    expected_performance: Dict[str, float]


@dataclass
class HumanModification:
    """A simulated human suggestion for an AI recommendation."""

    vessel_id: int
    original_position: float
    modified_position: float
    original_start: float
    modified_start: float
    original_cranes: int
    modified_cranes: int
    rationale: str
    contextual_factors: List[str]


@dataclass
class CollaborationMetrics:
    """Illustrative metrics; these are proxies, not empirical measurements."""

    ai_only_performance: float
    human_only_performance: float
    collaborative_performance: float
    customer_satisfaction: float
    decision_confidence: float
    learning_improvement: float


print("Data structures defined for the human–AI collaboration demonstration.")

In [ ]:
class HumanAISymbioticSystem:
    """Rule-based greedy recommender for the illustrative BAP-QCAP scenario."""

    def __init__(self, quay_length: float = 1200, num_cranes: int = 15):
        if quay_length <= 0 or num_cranes < 1:
            raise ValueError("Quay length must be positive and at least one crane must be available.")

        self.quay_length = quay_length
        self.num_cranes = num_cranes
        self.crane_productivity = 30  # TEU per hour per crane
        self.interference_factor = 0.1
        self.learning_rate = 0.1

        # These containers are reserved for possible future feedback records; this
        # demonstration does not use them to train or alter the recommendation rules.
        self.human_preference_patterns = defaultdict(list)
        self.contextual_factors = defaultdict(list)
        self.performance_history = []

    def generate_ai_recommendations(
        self, vessels: List[Vessel]
    ) -> List[AIRecommendation]:
        """Recommend assignments in priority-and-arrival order."""
        if not vessels:
            return []

        self._validate_vessels(vessels)
        sorted_vessels = sorted(vessels, key=lambda vessel: (vessel.priority, vessel.arrival_time))
        recommendations = []
        current_time = 0.0

        for vessel in sorted_vessels:
            # Earlier vessels finish before the next one starts, so the berth can
            # be reused; the position rule is a preference, not a concurrency check.
            berth_position = self._find_optimal_position(vessel, [])
            crane_count = self._calculate_optimal_cranes(vessel)
            start_time = max(vessel.arrival_time, current_time)
            confidence = self._calculate_confidence(vessel, berth_position, crane_count)
            reasoning = self._generate_reasoning(
                vessel, berth_position, crane_count, start_time
            )
            alternatives = self._generate_alternatives(
                vessel, berth_position, crane_count
            )
            expected_performance = self._calculate_expected_performance(
                vessel, berth_position, crane_count, start_time
            )

            recommendations.append(
                AIRecommendation(
                    vessel_id=vessel.id,
                    berth_position=berth_position,
                    start_time=start_time,
                    crane_count=crane_count,
                    confidence=confidence,
                    reasoning=reasoning,
                    alternatives=alternatives,
                    expected_performance=expected_performance,
                )
            )
            current_time = start_time + expected_performance["service_time"]

        return recommendations

    def _validate_vessels(self, vessels: List[Vessel]) -> None:
        """Reject inputs that make the demonstration's calculations undefined."""
        identifiers = [vessel.id for vessel in vessels]
        if len(set(identifiers)) != len(identifiers):
            raise ValueError("Vessel identifiers must be unique.")

        for vessel in vessels:
            if vessel.length <= 0 or vessel.length >= self.quay_length:
                raise ValueError("Each vessel length must be positive and shorter than the quay.")
            if vessel.workload < 0 or vessel.arrival_time < 0:
                raise ValueError("Workload and arrival time must be non-negative.")
            if vessel.priority not in (1, 2, 3):
                raise ValueError("Priority must be 1 (high), 2 (medium), or 3 (low).")
            if not 0 <= vessel.customer_value <= 1:
                raise ValueError("Customer value must lie between 0 and 1.")

    def _find_optimal_position(
        self, vessel: Vessel, used_positions: List[Tuple[float, float]]
    ) -> float:
        """Choose a 50-meter-grid position using a simple draft preference."""
        candidates = range(0, int(self.quay_length - vessel.length) + 1, 50)
        best_position = None
        best_score = float("inf")

        for position in candidates:
            vessel_range = (position, position + vessel.length)
            conflicts = sum(
                not (vessel_range[1] <= used_start or vessel_range[0] >= used_end)
                for used_start, used_end in used_positions
            )

            # The draft preference favors positions at or beyond 300 meters for
            # vessels with draft above 14 meters; it is a scenario rule, not bathymetry.
            draft_penalty = (
                -10 if vessel.draft > 14 and position >= 300 else 0
            )
            score = conflicts + draft_penalty

            if score < best_score:
                best_score = score
                best_position = position

        if best_position is None:
            raise ValueError("No berth-position candidate is available for this vessel.")
        return float(best_position)

    def _effective_productivity(self, cranes: int) -> float:
        """Apply the same interference and high-crane penalty throughout the model."""
        if cranes < 1 or cranes > self.num_cranes:
            raise ValueError("Crane count must be within the available crane limit.")

        productivity = (
            cranes
            * self.crane_productivity
            * (1 - self.interference_factor * (cranes - 1))
        )
        if cranes > 3:
            productivity *= 0.9
        return productivity

    def _calculate_optimal_cranes(self, vessel: Vessel) -> int:
        """Select the allowed crane count with the greatest modeled productivity."""
        del vessel  # The rule uses system productivity settings, not vessel attributes.
        max_cranes = min(self.num_cranes, 5)
        return max(
            range(1, max_cranes + 1),
            key=self._effective_productivity,
        )

    def _calculate_confidence(
        self, vessel: Vessel, position: float, cranes: int
    ) -> float:
        """Return a rule-based score; it is not a calibrated probability."""
        confidence = 0.8
        if 200 <= vessel.length <= 350:
            confidence += 0.1
        if cranes >= 4:
            confidence -= 0.15
        if 200 <= position <= 800:
            confidence += 0.05
        if vessel.priority == 1:
            confidence += 0.05
        return min(max(confidence, 0.3), 0.95)

    def _generate_reasoning(
        self, vessel: Vessel, position: float, cranes: int, start_time: float
    ) -> List[str]:
        """Explain the main rules that determine a recommendation."""
        reasoning = [
            f"Position {position:.0f} m selected by the berth-position preference for a {vessel.length:.0f} m vessel."
        ]

        if vessel.draft > 14 and position >= 300:
            reasoning.append(
                f"The scenario's deep-water preference applies to the {vessel.draft:.1f} m draft."
            )

        productivity = self._effective_productivity(cranes)
        reasoning.append(
            f"{cranes} cranes provide modeled effective productivity of {productivity:.1f} TEU/hour."
        )
        if cranes > 3:
            reasoning.append("The productivity estimate includes the additional penalty for more than three cranes.")

        waiting_time = max(0.0, start_time - vessel.arrival_time)
        if waiting_time > 0:
            reasoning.append(
                f"The sequential schedule includes {waiting_time:.1f} hours of waiting after arrival."
            )
        else:
            reasoning.append("The vessel can start service at its arrival time under the sequential schedule.")
        return reasoning

    def _generate_alternatives(
        self, vessel: Vessel, position: float, cranes: int
    ) -> List[Dict]:
        """Compare simple one-vessel position and crane-count alternatives."""
        baseline = self._calculate_expected_performance(
            vessel, position, cranes, vessel.arrival_time
        )
        position_candidates = list(
            range(0, int(self.quay_length - vessel.length) + 1, 50)
        )
        alternative_position = next(
            (candidate for candidate in position_candidates if candidate != position),
            position,
        )
        position_performance = self._calculate_expected_performance(
            vessel, alternative_position, cranes, vessel.arrival_time
        )
        alternatives = [
            {
                "type": "position_change",
                "position": alternative_position,
                "cranes": cranes,
                "impact": (
                    f"One-vessel total time: {position_performance['total_time']:.1f} h; "
                    f"change from baseline: {position_performance['total_time'] - baseline['total_time']:+.1f} h."
                ),
            }
        ]

        alternative_cranes = cranes - 1 if cranes > 1 else min(cranes + 1, self.num_cranes, 5)
        crane_performance = self._calculate_expected_performance(
            vessel, position, alternative_cranes, vessel.arrival_time
        )
        alternatives.append(
            {
                "type": "crane_change",
                "position": position,
                "cranes": alternative_cranes,
                "impact": (
                    f"One-vessel total time: {crane_performance['total_time']:.1f} h; "
                    f"change from baseline: {crane_performance['total_time'] - baseline['total_time']:+.1f} h."
                ),
            }
        )
        return alternatives

    def _calculate_expected_performance(
        self, vessel: Vessel, position: float, cranes: int, start_time: float
    ) -> Dict[str, float]:
        """Calculate workload, waiting, service, and illustrative cost estimates."""
        del position  # Position has no modeled effect on service rate in this prototype.
        productivity = self._effective_productivity(cranes)
        service_time = vessel.workload / productivity if vessel.workload else 0.0
        waiting_time = max(0.0, start_time - vessel.arrival_time)
        total_time = waiting_time + service_time
        priority_rate = {1: 1000, 2: 500, 3: 200}[vessel.priority]

        return {
            "service_time": service_time,
            "waiting_time": waiting_time,
            "total_time": total_time,
            "productivity": productivity,
            "estimated_cost": total_time * priority_rate,
        }


print("Greedy recommendation system defined.")

In [ ]:
class HumanExpertInterface:
    """Simulate contextual review of AI recommendations."""

    def __init__(self):
        self.expertise_areas = [
            "customer relations",
            "weather planning",
            "equipment maintenance",
            "labor management",
        ]
        self.strategic_priorities = {
            "customer_satisfaction": 0.3,
            "operational_efficiency": 0.25,
            "resource_utilization": 0.2,
            "safety_compliance": 0.15,
            "cost_optimization": 0.1,
        }

    def review_ai_recommendations(
        self,
        recommendations: List[AIRecommendation],
        vessels: List[Vessel],
        num_cranes: int,
    ) -> List[HumanModification]:
        """Return simulated suggestions while preserving the recommendation record."""
        vessel_by_id = {vessel.id: vessel for vessel in vessels}
        modifications = []

        for recommendation in recommendations:
            vessel = vessel_by_id[recommendation.vessel_id]
            modification = self._apply_human_judgment(
                recommendation, vessel, num_cranes
            )
            if modification is not None:
                modifications.append(modification)
        return modifications

    def _apply_human_judgment(
        self,
        recommendation: AIRecommendation,
        vessel: Vessel,
        num_cranes: int,
    ) -> Optional[HumanModification]:
        """Choose one contextual suggestion using the seeded random generator."""
        if random.random() >= 0.3:
            return None

        modified_position = recommendation.berth_position
        modified_start = recommendation.start_time
        modified_cranes = recommendation.crane_count

        if vessel.customer_value > 0.8 and recommendation.confidence < 0.85:
            modified_start = max(
                vessel.arrival_time, recommendation.start_time - 0.5
            )
            rationale = "Requested earlier consideration for a high-value customer."
            factors = ["Customer retention strategy", "Premium service agreement"]
        elif vessel.priority == 1 and random.random() < 0.4:
            modified_cranes = min(recommendation.crane_count + 1, num_cranes, 5)
            rationale = "Requested a weather contingency review for a high-priority vessel."
            factors = ["Weather forecast uncertainty", "Storm preparation"]
        elif recommendation.berth_position > 800 and random.random() < 0.3:
            modified_position = max(0.0, recommendation.berth_position - 200)
            rationale = "Requested review of the planned maintenance area."
            factors = ["Crane maintenance schedule", "Equipment availability"]
        else:
            modified_start = recommendation.start_time + 0.25
            rationale = "Requested review of shift timing and crew availability."
            factors = ["Shift change timing", "Crew productivity patterns"]

        return HumanModification(
            vessel_id=vessel.id,
            original_position=recommendation.berth_position,
            modified_position=modified_position,
            original_start=recommendation.start_time,
            modified_start=modified_start,
            original_cranes=recommendation.crane_count,
            modified_cranes=modified_cranes,
            rationale=rationale,
            contextual_factors=factors,
        )

    def provide_explanation_feedback(
        self, recommendation: AIRecommendation, feedback_score: int
    ) -> None:
        """Placeholder for feedback capture; no learning update is implemented."""
        if not 1 <= feedback_score <= 5:
            raise ValueError("Feedback score must be an integer from 1 to 5.")
        del recommendation


print("Human expert review interface defined.")

In [ ]:
class CollaborativeOptimizer:
    """Coordinate AI recommendations, human review, and metric tracking."""

    def __init__(self):
        self.ai_system = HumanAISymbioticSystem()
        self.human_expert = HumanExpertInterface()
        self.learning_history = []

    def optimize_collaboratively(
        self, vessels: List[Vessel]
    ) -> Tuple[List[AIRecommendation], List[HumanModification], CollaborationMetrics]:
        """Run the five-stage recommendation and review workflow."""
        ai_recommendations = self.ai_system.generate_ai_recommendations(vessels)
        human_modifications = self.human_expert.review_ai_recommendations(
            ai_recommendations, vessels, self.ai_system.num_cranes
        )
        collaborative_solution = self._apply_modifications(
            ai_recommendations, human_modifications, vessels
        )
        metrics = self._calculate_collaboration_metrics(
            ai_recommendations,
            human_modifications,
            collaborative_solution,
            vessels,
        )
        self._update_learning_models(human_modifications, metrics)
        return collaborative_solution, human_modifications, metrics

    def _apply_modifications(
        self,
        recommendations: List[AIRecommendation],
        modifications: List[HumanModification],
        vessels: List[Vessel],
    ) -> List[AIRecommendation]:
        """Apply suggestions and rebuild a feasible sequential timing sequence."""
        vessel_by_id = {vessel.id: vessel for vessel in vessels}
        modification_by_id = {
            modification.vessel_id: modification for modification in modifications
        }
        collaborative = []
        current_time = 0.0

        for recommendation in recommendations:
            vessel = vessel_by_id[recommendation.vessel_id]
            modification = modification_by_id.get(recommendation.vessel_id)
            position = recommendation.berth_position
            cranes = recommendation.crane_count
            requested_start = recommendation.start_time
            reasoning = list(recommendation.reasoning)
            confidence = recommendation.confidence

            if modification is not None:
                position = modification.modified_position
                cranes = modification.modified_cranes
                requested_start = modification.modified_start
                reasoning.append(f"Human review: {modification.rationale}")
                confidence = min(0.95, confidence + 0.1)

            if not 0 <= position <= self.ai_system.quay_length - vessel.length:
                raise ValueError("A berth-position suggestion lies outside the quay.")
            if not 1 <= cranes <= self.ai_system.num_cranes:
                raise ValueError("A crane-count suggestion exceeds available resources.")

            # The start time respects arrival, the preceding service completion,
            # and any later start requested during human review.
            start_time = max(vessel.arrival_time, current_time, requested_start)
            performance = self.ai_system._calculate_expected_performance(
                vessel, position, cranes, start_time
            )
            collaborative.append(
                AIRecommendation(
                    vessel_id=recommendation.vessel_id,
                    berth_position=position,
                    start_time=start_time,
                    crane_count=cranes,
                    confidence=confidence,
                    reasoning=reasoning,
                    alternatives=recommendation.alternatives,
                    expected_performance=performance,
                )
            )
            current_time = start_time + performance["service_time"]

        return collaborative

    def _calculate_collaboration_metrics(
        self,
        ai_recommendations: List[AIRecommendation],
        human_modifications: List[HumanModification],
        collaborative_solution: List[AIRecommendation],
        vessels: List[Vessel],
    ) -> CollaborationMetrics:
        """Compute explicitly defined proxy metrics for this demonstration."""
        if not vessels:
            return CollaborationMetrics(0, 0, 0, 0.85, 0, 0)

        ai_total_time = sum(
            recommendation.expected_performance["total_time"]
            for recommendation in ai_recommendations
        )
        collaborative_total_time = sum(
            recommendation.expected_performance["total_time"]
            for recommendation in collaborative_solution
        )
        ai_performance = 100 / (1 + ai_total_time)
        collaborative_performance = 100 / (1 + collaborative_total_time)

        # Human-only performance is an assumed comparison proxy, not a measured result.
        human_performance = ai_performance * 0.85
        satisfaction = 0.85 + (len(human_modifications) / len(vessels)) * 0.1
        average_confidence = sum(
            recommendation.confidence for recommendation in collaborative_solution
        ) / len(collaborative_solution)

        # This is observed score change relative to the first recorded session.
        # It does not imply that the recommendation policy learned or improved.
        learning_improvement = 0.0
        if self.learning_history:
            learning_improvement = (
                collaborative_performance
                - self.learning_history[0]["collaborative_performance"]
            )

        return CollaborationMetrics(
            ai_only_performance=ai_performance,
            human_only_performance=human_performance,
            collaborative_performance=collaborative_performance,
            customer_satisfaction=satisfaction,
            decision_confidence=average_confidence * 100,
            learning_improvement=learning_improvement,
        )

    def _update_learning_models(
        self,
        modifications: List[HumanModification],
        metrics: CollaborationMetrics,
    ) -> None:
        """Record a session for analysis; this method does not train a model."""
        self.learning_history.append(
            {
                "modifications_count": len(modifications),
                "collaborative_performance": metrics.collaborative_performance,
                "customer_satisfaction": metrics.customer_satisfaction,
                "timestamp": time.time(),
            }
        )


print("Collaborative workflow defined.")

In [ ]:
# Vessel data for the 72-hour peak-season demonstration.
vessels = [
    Vessel(1, "Maersk Shanghai", 300, 800, 8.0, 14.2, 1, 0.9),
    Vessel(2, "MSC Mediterranean", 350, 1200, 10.0, 15.1, 1, 0.85),
    Vessel(3, "COSCO Development", 400, 1500, 12.0, 15.8, 2, 0.7),
    Vessel(4, "ONE Compass", 250, 600, 14.0, 13.5, 2, 0.6),
    Vessel(5, "HMM Algeciras", 300, 900, 16.0, 14.8, 1, 0.95),
    Vessel(6, "Evergreen Ace", 280, 750, 18.0, 14.0, 3, 0.4),
    Vessel(7, "CMA CGM Bougainville", 350, 1100, 20.0, 15.2, 2, 0.65),
    Vessel(8, "OOCL Hong Kong", 380, 1400, 22.0, 15.6, 1, 0.88),
    Vessel(9, "Yang Ming Hope", 220, 500, 24.0, 13.2, 3, 0.35),
    Vessel(10, "PIL Endurance", 260, 700, 26.0, 13.8, 2, 0.55),
    Vessel(11, "Wan Hai 501", 200, 450, 28.0, 12.8, 3, 0.3),
    Vessel(12, "Zim Liberia", 320, 850, 30.0, 14.5, 2, 0.6),
    Vessel(13, "Hapag-Lloyd New York", 340, 1050, 32.0, 15.0, 1, 0.82),
    Vessel(14, "X-Press Pearl", 240, 650, 34.0, 13.6, 3, 0.38),
    Vessel(15, "Seaspan Energy", 290, 780, 36.0, 14.3, 2, 0.58),
]

print(f"Created {len(vessels)} vessels for the 72-hour peak-season scenario.")
print("First five vessel records:")
for vessel in vessels[:5]:
    print(
        f"  {vessel.name}: {vessel.length} m, {vessel.workload} TEU, "
        f"priority {vessel.priority}, customer-value score {vessel.customer_value:.2f}"
    )
print("The remaining ten records are available in the vessels list.")

In [ ]:
optimizer = CollaborativeOptimizer()

print("=== Human–AI Collaboration for BAP-QCAP ===")
print(f"Vessels: {len(vessels)}")
print(f"Quay length: {optimizer.ai_system.quay_length:.0f} m")
print(f"Available cranes: {optimizer.ai_system.num_cranes}")

collaborative_solution, human_modifications, metrics = (
    optimizer.optimize_collaboratively(vessels)
)

print("\n=== Illustrative collaboration metrics ===")
print(f"AI-only proxy score: {metrics.ai_only_performance:.2f}")
print(f"Human-only proxy score (assumed): {metrics.human_only_performance:.2f}")
print(f"Collaborative proxy score: {metrics.collaborative_performance:.2f}")
print(
    "Collaborative minus AI-only proxy score: "
    f"{metrics.collaborative_performance - metrics.ai_only_performance:+.2f}"
)
print(f"Customer-satisfaction proxy: {metrics.customer_satisfaction:.1%}")
print(f"Mean heuristic confidence: {metrics.decision_confidence:.1f}/100")
print(f"Human suggestions recorded: {len(human_modifications)} of {len(vessels)} vessels")

In [ ]:
vessel_by_id = {vessel.id: vessel for vessel in vessels}
print("=== Recommendations with explanations ===\n")

for recommendation in collaborative_solution[:5]:
    vessel = vessel_by_id[recommendation.vessel_id]
    print(f"Vessel {recommendation.vessel_id}: {vessel.name}")
    print(
        f"  Assignment: position {recommendation.berth_position:.0f} m, "
        f"start {recommendation.start_time:.2f} h, "
        f"{recommendation.crane_count} cranes"
    )
    print(f"  Heuristic confidence score: {recommendation.confidence:.1%}")
    print(
        "  Estimated total time: "
        f"{recommendation.expected_performance['total_time']:.2f} h"
    )
    print("  Reasoning:")
    for reason in recommendation.reasoning:
        print(f"    - {reason}")
    print("  One-vessel alternatives:")
    for alternative in recommendation.alternatives:
        print(f"    - {alternative['type']}: {alternative['impact']}")
    print()

print("Detailed recommendation records for all vessels are in collaborative_solution.")

In [ ]:
print("=== Simulated human review suggestions ===\n")

if human_modifications:
    for modification in human_modifications:
        vessel = vessel_by_id[modification.vessel_id]
        print(f"Vessel {modification.vessel_id}: {vessel.name}")
        print(
            f"  Suggested position: {modification.original_position:.0f} m "
            f"to {modification.modified_position:.0f} m"
        )
        print(
            f"  Suggested start: {modification.original_start:.2f} h "
            f"to {modification.modified_start:.2f} h"
        )
        print(
            f"  Suggested crane count: {modification.original_cranes} "
            f"to {modification.modified_cranes}"
        )
        print(f"  Rationale: {modification.rationale}")
        print(f"  Context: {', '.join(modification.contextual_factors)}\n")
else:
    print("No suggestions were generated in this seeded simulation.")

modification_rate = len(human_modifications) / len(vessels) if vessels else 0.0
print(f"Suggestion rate: {modification_rate:.1%}")
print("Suggestions are simulated inputs; the resulting schedule is rebuilt sequentially.")

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# The three scores are illustrative proxies with different assumptions.
methods = ["AI-only proxy", "Human-only proxy", "Collaborative proxy"]
performance = [
    metrics.ai_only_performance,
    metrics.human_only_performance,
    metrics.collaborative_performance,
]
performance_colors = ["lightblue", "lightgreen", "gold"]
bars = ax1.bar(methods, performance, color=performance_colors, edgecolor="black")
ax1.set_title("Illustrative Performance Proxies")
ax1.set_ylabel("Score (higher is better; notebook-defined scale)")
ax1.tick_params(axis="x", rotation=15)
ax1.set_ylim(0, max(performance, default=1) * 1.2 + 1)
for bar, score in zip(bars, performance):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{score:.2f}",
        ha="center",
        va="bottom",
    )

confidence_scores = [item.confidence for item in collaborative_solution]
if confidence_scores:
    ax2.hist(confidence_scores, bins=10, color="skyblue", edgecolor="black")
    ax2.axvline(
        np.mean(confidence_scores),
        color="red",
        linestyle="--",
        label=f"Mean: {np.mean(confidence_scores):.2f}",
    )
    ax2.legend()
ax2.set_title("Heuristic Confidence Scores")
ax2.set_xlabel("Score (not a calibrated probability)")
ax2.set_ylabel("Number of vessels")

if human_modifications:
    modification_types = [item.rationale.split()[0] for item in human_modifications]
    counts = pd.Series(modification_types).value_counts()
    pie_colors = plt.cm.Set3(np.linspace(0, 1, len(counts)))
    ax3.pie(
        counts.values,
        labels=counts.index,
        autopct="%1.1f%%",
        colors=pie_colors,
        startangle=90,
    )
else:
    ax3.text(
        0.5,
        0.5,
        "No suggestions generated",
        ha="center",
        va="center",
        transform=ax3.transAxes,
    )
ax3.set_title("Simulated Suggestion Categories")

quality_labels = ["Customer satisfaction\nproxy", "Mean confidence"]
quality_values = [
    metrics.customer_satisfaction * 100,
    metrics.decision_confidence,
]
quality_bars = ax4.bar(
    quality_labels,
    quality_values,
    color=["lightcoral", "lightsteelblue"],
    edgecolor="black",
)
ax4.set_title("Additional Notebook-Defined Indicators")
ax4.set_ylabel("Score (0–100 scale)")
ax4.set_ylim(0, max(quality_values, default=1) * 1.2 + 1)
for bar, value in zip(quality_bars, quality_values):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}",
        ha="center",
        va="bottom",
    )

fig.suptitle("Human–AI Collaboration Demonstration", fontsize=16)
fig.tight_layout()
plt.show()

print("Visualizations show notebook-defined proxies, not validated operational outcomes.")

In [ ]:
print("=== Repeated-session analysis ===\n")

# Reuse one optimizer so that session records accumulate. Its recommendation
# rules remain unchanged; this experiment records variation rather than learning.
session_optimizer = CollaborativeOptimizer()
learning_sessions = []

for session_number in range(5):
    session_vessels = [
        Vessel(
            vessel.id,
            vessel.name,
            vessel.length,
            vessel.workload,
            max(0.0, vessel.arrival_time + random.uniform(-2, 2)),
            vessel.draft,
            vessel.priority,
            vessel.customer_value,
        )
        for vessel in vessels
    ]
    _, session_modifications, session_metrics = (
        session_optimizer.optimize_collaboratively(session_vessels)
    )
    learning_sessions.append(
        {
            "session": session_number + 1,
            "performance": session_metrics.collaborative_performance,
            "satisfaction": session_metrics.customer_satisfaction,
            "confidence": session_metrics.decision_confidence,
            "suggestions": len(session_modifications),
        }
    )

session_numbers = [item["session"] for item in learning_sessions]
performances = [item["performance"] for item in learning_sessions]
satisfactions = [item["satisfaction"] * 100 for item in learning_sessions]
confidences = [item["confidence"] for item in learning_sessions]
suggestion_counts = [item["suggestions"] for item in learning_sessions]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.plot(session_numbers, performances, "o-", label="Collaborative proxy score")
ax1.plot(session_numbers, satisfactions, "s-", label="Customer-satisfaction proxy")
ax1.plot(session_numbers, confidences, "^-", label="Mean heuristic confidence")
ax1.set_title("Metrics Across Simulated Sessions")
ax1.set_xlabel("Session")
ax1.set_ylabel("Notebook-defined score")
ax1.set_xticks(session_numbers)
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.bar(session_numbers, suggestion_counts, color="lightsteelblue", edgecolor="black")
ax2.set_title("Human Suggestions by Session")
ax2.set_xlabel("Session")
ax2.set_ylabel("Number of simulated suggestions")
ax2.set_xticks(session_numbers)
ax2.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

print("Session summary:")
for item in learning_sessions:
    print(
        f"  Session {item['session']}: proxy score {item['performance']:.2f}, "
        f"satisfaction proxy {item['satisfaction']:.1%}, "
        f"mean confidence {item['confidence']:.1f}/100, "
        f"suggestions {item['suggestions']}"
    )

observed_change = performances[-1] - performances[0]
print(f"Observed first-to-last proxy-score change: {observed_change:+.2f} points")
print("This change reflects scenario variation, not demonstrated policy learning.")

## Summary and interpretation

The notebook implements a transparent sequence of rules for generating berth and crane recommendations, followed by simulated human review. The AI procedure sorts vessels by priority and arrival time, uses a draft-related berth-position preference, evaluates crane counts by modeled productivity, and schedules vessels sequentially. Human suggestions are then applied and the service sequence is rebuilt so that each start respects arrival time and the preceding vessel's modeled completion.

### How to interpret the indicators

- The **AI-only** and **collaborative** scores are transformations of the respective total-time estimates. A higher score corresponds to a lower modeled total time under the notebook's formula.
- The **human-only** score is defined as 85% of the AI-only score. It is an explicit illustrative assumption, not the result of a separately implemented human scheduling process.
- The **customer-satisfaction proxy** rises with the number of simulated human suggestions by design. It is not a survey or observed customer outcome.
- The **confidence score** is a hand-coded heuristic. It should not be interpreted as the probability that an assignment is correct.
- The repeated-session plot shows scenario-to-scenario variation. The code records session outcomes but does not update its decision rules, so a change between sessions is not evidence of learning.

### Modeling boundaries

The sequential schedule prevents temporal overlap in this simplified scenario, but it does not solve the full integrated berth and crane allocation problem. Position choice is not coupled to water depth data or service productivity, and the model does not allocate cranes across simultaneously serviced vessels. The illustrative cost and performance measures are not optimized as a formal objective. These boundaries are important when distinguishing a teaching demonstration from an operational planning model.

### Key lessons

1. Explainable rules make recommendations easier to inspect, but transparency alone does not establish that a recommendation is optimal or feasible under every real-world constraint.
2. Human context can be represented as explicit suggestions, while schedule feasibility still needs to be checked after those suggestions are applied.
3. A metric is meaningful only when its formula and assumptions are stated. Simulated proxies must not be presented as measured performance.
4. A recorded feedback history is not a learning system unless the feedback changes a model or decision rule and that change is evaluated.

Natural extensions include adding operational constraints and data, defining an empirically meaningful objective, and validating recommendations against real schedules. Those extensions require a fuller model than the illustrative procedure implemented here.